In [ ]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import os


def phases(iq: np.ndarray,start:int):
    list = []
    for i in range(start,len(iq)-2,6):
        list.append(
            np.angle(complex(iq[i],iq[i+1]))
        )
    return np.array(list)


def single_antenna_to_dataframe(read_fp: str, write_fp=None) -> pd.DataFrame:
    '''Transforma uma captura com uma ?nica antena em um df '''
    raw =pd.read_json(read_fp)
    df= list()
    for group in raw[1]:
        group["Angle"] = read_fp.split('_')[1].split("-")[0]
        group["Distance"] = read_fp.split('_')[-1].split("-")[-1].split(".")[0]
        df.append(group)
    df = pd.DataFrame(df)
    df = df.join(pd.DataFrame(df["aoa"].to_list()))
    df.pop("aoa")

    if write_fp is not None:
        df.to_json(write_fp)
    else:
        return df

data = pd.DataFrame()

for fp in os.listdir("./data"):
    df = single_antenna_to_dataframe("./data/" + fp)
    data = pd.concat( (df, data))

data =data.reset_index()

In [ ]:
data.to_json("./data.json")

In [ ]:
data

In [ ]:
tdata = []
tlabels = []

for _, row in data.iterrows():
    for i in range(16,len(row["iq"])-6,3*2):
        tdata.append("frequency",row["iq"][i:i+6])
        tlabels.append(row["Angle"])
len(tdata) == len(tlabels)

In [ ]:
len(tdata)

In [ ]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, mean_absolute_error, mean_squared_error

#Loading the Iris dataset

label_mapper = lambda x: {"0":0,
                "15":1,
                "30":2,
                "45":3,
                "60":4,
                "75":5,
                "90":6,
                "105":7,
                "120":8,
                "135":9,
                "150":10,
                "165":11,
                "180":12,
                }.get(str(x))

X, y = tdata, [label_mapper(x) for x in tlabels]

#Splitting the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)

#Creating an XGBoost classifier
model = xgb.XGBClassifier(enable_categorical=True)

#Training the model on the training data
model.fit(X_train, y_train)


In [ ]:
model.save_model(fname="./classifier.json")


In [ ]:
#Making predictions on the test set
predictions = model.predict(X_test)

#Calculating accuracy
accuracy = accuracy_score(y_test, predictions)

print("Accuracy:", accuracy)
print("\nClassification Report:")
print(classification_report(y_test, predictions, target_names={"0","15","30","45","60","75","90","105","120","135","150","165","180"}))

# Regressor

In [ ]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, mean_absolute_error, mean_squared_error

#Loading the Iris dataset

X, y = np.array(tdata), [int(x) for x in tlabels]

#Splitting the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)

#Creating an XGBoost classifier
model = xgb.XGBRegressor(eval_metric=mean_absolute_error)

#Training the model on the training data
model.fit(X_train, y_train)
model.save_model(fname="./regressor.json")

In [ ]:
#Making predictions on the test set
predictions = model.predict(X_test)

mae = mean_absolute_error(y_test, predictions)
print('Erro M?dio Absoluto:', mae)

# Calcular MSE
mse = mean_squared_error(y_test, predictions)
print('Erro Quadr?tico M?dio:', mse)

# Calcular RMSE
rmse = mean_squared_error(y_test, predictions, squared=False)
print('Raiz do Erro Quadr?tico M?dio:', rmse)

In [ ]:
i=np.random.randint(len(X_test))
predictions = model.predict([X_test[i]])
print(f"{predictions=} \n answer={y_test[i]}")


In [ ]:
predictions = []

for i in range(len(X_test)):
    predictions.append(model.predict([X_test[i]]))

In [ ]:
diff =np.array(y_test)-np.array(predictions)

In [ ]:
from matplotlib import pyplot as plt

plt.plot(y_test,diff,".")